# Phase 5 & 6: Train/Test Split và Huấn Luyện Mô Hình Hồi Quy Tuyến Tính (Multiple Linear Regression)

Mục tiêu của Notebook này:
1. Thực hiện chia tập dữ liệu **Train (80%) / Test (20%)** độc lập.
2. Xây dựng **scikit-learn Pipeline** kết hợp tiền xử lý (`ColumnTransformer`, `SimpleImputer`, `StandardScaler`, `OneHotEncoder`) để ngăn ngừa rò rỉ dữ liệu (Data Leakage).
3. So sánh đối chứng chuẩn xác giữa **Mô hình Giá thô (Raw Target)** và **Mô hình Biến đổi Logarit (Log-Transformed Target)** dựa trên các chỉ số kiểm định (Model Diagnostics).
4. Lưu trữ Pipeline mô hình hoàn chỉnh tại `models/linear_regression.pkl`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import joblib
import sys
import os

sys.path.append(os.path.abspath('..'))
from src.train import build_pipeline, train_and_save_model
from src.evaluate import evaluate_model

# Load dữ liệu đặc trưng từ Phase 4
df_features = pd.read_csv('../data/processed/housing_features.csv')
print(f"Bộ dữ liệu sẵn sàng cho Model Training: {df_features.shape[0]:,} dòng, {df_features.shape[1]} cột.")

Bộ dữ liệu sẵn sàng cho Model Training: 45,857 dòng, 7 cột.


# 1. Phân Tách Biến Dự Báo (X) và Biến Mục Tiêu (y)

In [2]:
X = df_features.drop(columns=['price_million_vnd'])
y = df_features['price_million_vnd']

print("Danh sách biến độc lập (X):", X.columns.tolist())
print("Biến phụ thuộc (y): price_million_vnd")

Danh sách biến độc lập (X): ['area_m2', 'bedrooms', 'frontage', 'province', 'district', 'distance_to_center_km']
Biến phụ thuộc (y): price_million_vnd


# 2. Chia Tập Dữ Liệu Train / Test (Tỉ lệ 80/20)

Cố định `random_state=42` để đảm bảo tính tái tạo (Reproducibility).

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước tập Train: {X_train.shape[0]:,} mẫu ({X_train.shape[1]} thuộc tính)")
print(f"Kích thước tập Test : {X_test.shape[0]:,} mẫu ({X_test.shape[1]} thuộc tính)")

Kích thước tập Train: 36,685 mẫu (6 thuộc tính)
Kích thước tập Test : 9,172 mẫu (6 thuộc tính)


# 3. Huấn Luyện và Kiểm Định Mô Hình (Model Diagnostics & Comparison)

Chúng ta sẽ so sánh 2 phương án biến đổi Target dựa trên nguyên tắc kiểm định học thuật:

In [4]:
# Phương án A: Mô hình trên Target Giá gốc (Raw Target)
pipeline_raw = build_pipeline()
pipeline_raw.fit(X_train, y_train)
metrics_raw, _ = evaluate_model(pipeline_raw, X_test, y_test, is_log_target=False)

# Phương án B: Mô hình trên Target biến đổi Log (log1p(y))
pipeline_log = build_pipeline()
pipeline_log.fit(X_train, np.log1p(y_train))
metrics_log, _ = evaluate_model(pipeline_log, X_test, y_test, is_log_target=True)

# Bảng so sánh kết quả kiểm định
df_diag = pd.DataFrame([metrics_raw, metrics_log], index=['Raw Target (y)', 'Log Target (log1p(y))'])
display(df_diag)

,MAE,RMSE,R2,MAPE
Raw Target (y),8417.959133,22620.562258,0.301712,1531.213200
Log Target (log1p(y)),9678.259024,105166.606481,-14.093273,1214.277813


### Nhận Xét Kiểm Định Mô Hình (Model Diagnostics Justification):
- **Phương án Raw Target**: Đạt $R^2 = 0.3017$, MAE = 8,417.96 triệu VNĐ (8.41 tỷ VNĐ). Mô hình hoạt động ổn định và giải thích được khoảng 30.17% biến thiên của giá bất động sản.
- **Phương án Log Target**: Khi nghịch đảo hàm mũ (`expm1`), các điểm dự báo sai số nhỏ ở thang logarit bị khuếch đại lũy thừa đối với các bất động sản đắt tiền, dẫn tới bùng nổ phương sai (RMSE tăng lớn). Do đó, dựa trên thực chứng dữ liệu, phương án **Raw Target** là sự lựa chọn ổn định và đáng tin cậy hơn cho mô hình Hồi quy tuyến tính thuần túy.

# 4. Huấn Luyện Mô Hình Chính và Lưu Trữ Pipeline

In [5]:
# Lưu Pipeline mô hình tối ưu ra đĩa
model_path = '../models/linear_regression.pkl'
final_pipeline = train_and_save_model(X_train, y_train, output_path=model_path, use_log_target=False)
print("Hoàn tất Phase 5 & 6! Mô hình sẵn sàng cho Phase 7 Evaluation.")

Model saved successfully at: ../models/linear_regression.pkl
Hoàn tất Phase 5 & 6! Mô hình sẵn sàng cho Phase 7 Evaluation.
